In [ ]:
# %matplotlib inline
import os
import csv
from glob import glob
import json
import logging
import math
from pathlib import Path
import subprocess

import numpy as np

import plotly
import plotly.graph_objs as go
import matplotlib
matplotlib.use('qtagg')
import matplotlib.pyplot as plt
plotly.offline.init_notebook_mode(connected=True)

import spiceypy as spice

import ale
from ale.drivers.clipper_drivers import ClipperEISWACPBIsisLabelNaifSpiceDriver

import pyspiceql

import kalasiris
from kalasiris import isis
logging.basicConfig(level=logging.INFO)
from knoten.utils import reproject

# What is this Notebook?

The notebook allows for the generation of synthetic Europa Clipper WAC Pushbroom data. Originally built against unreleased label files and the preliminary kernels. Then update in May 2026 for use with publically available kernels from naif (https://naif.jpl.nasa.gov/pub/naif/EUROPACLIPPER/) and initial label files.

## How to use it

### Data Download
Anyone who wants to use this notebook will need a few things. First, you will need to download the clipper kernels from the naif link above under "kernels". This can be accessed directly using things like `wget` and `curl`, but the easiest way to download the data to work with the notebook is through [`downloadIsisData`](https://github.com/DOI-USGS/ISIS3/blob/dev/isis/scripts/downloadIsisData) and usage of that script can be found [here](https://astrogeology.usgs.gov/docs/how-to-guides/environment-setup-and-maintenance/isis-data-area/)

Once you have the data downloaded, you will need to point the notebook to that data by setting the `isis_data` variable below.

### Isis install
You will also need an ISIS installation to point the `ISISROOT` variable to. Follow the directions on from [here](https://astrogeology.usgs.gov/docs/how-to-guides/environment-setup-and-maintenance/installing-isis-via-anaconda/) on installing ISIS. This notebook also was created using ISIS 10.0.0_RC2. It's likely that any ISIS 10.0.X version will work. Once you have an ISIS installation, you will need to change the path for `ISISROOT` to the ISIS installation. If it's installed in a conda environment you can run `echo $CONDA_PREFIX` while the ISIS conda environment is active and that will be the `ISISROOT` path that you set below.

In [ ]:
isis_data = "/Path/to/isis_data/"
os.environ["ISISDATA"] = isis_data
os.environ["ISISROOT"] = "/Path/to/miniforge3/envs/isis10.0.0_RC2/"

In [ ]:
# Define various planet radii as spheres
mars_radii = np.array([3396.2, 3396.2, 3396.2])
europa_radii = np.array([1560.8, 1560.8, 1560.8])

# Get the EIS cube
label_path = "./"
clipper_cub = os.path.join(label_path, "EIS000XXX_2032116T234928_0000C35F-WAC-PUSHB-IMG_RAW_V1.LBL")

# Edit the value here for different targets
isis.editlab(from_=clipper_cub, 
             options="SETKEY", 
             grpname="Instrument", 
             keyword="TargetName", 
             value="Mars")

# Build kernel set
kernels = ['base/kernels/pck/pck00009.tpc',
           'base/kernels/spk/de430.bsp',
           'base/kernels/spk/mar097.bsp',
           'base/kernels/lsk/naif0012.tls',
           'clipper/kernels/sclk/europaclipper_00291.tsc',
           'clipper/kernels/ik/clipper_eis_v07.ti',
           'clipper/kernels/fk/clipper_v17.tf',
           'clipper/kernels/iak/clipperAddendum001.ti']
kernels = [os.path.join(isis_data, kernel) for kernel in kernels]

eis_kernels = kernels
eis_kernels

In [ ]:
def normalize(a):
    a = np.array(a)
    return a/np.linalg.norm(a)

# Degree coords to center the approach on
latitude = -5
longitude = 75

scale = mars_radii[0]/europa_radii[0]

# Height in km(?)
height = 50 * scale

# Direction of flight use "east" and "west"
# "east" is flying west toward east
# "west" is flygin east toward west
direction = "east"

# spacecraft speed in km/s
speed = 4.5 * scale

# +/- time from closest approach.
# Total time = start + (time_offset * 2)
time_offset = 500
# time_offset = time_offset * 2

# Radius definition to use
radii = mars_radii

# Compute and normalize nadir vector at closest approach
y, x, z = reproject([longitude, latitude, height], radii[0], radii[2], "latlong", "geocent")
spacecraft_point = np.array([x, y, z])
nadir_vector = normalize(spacecraft_point)

# Create a parallel trajectory that is perpendicular with nadir and parallel
# to the x, y axis
normal = np.array([0, 0, 1])
perpendicular_nadir = np.cross(nadir_vector, normal)

# compute fly_distance in km given speed and expected time from
# closest approach
fly_distance = speed * time_offset

# Compute the ending X,Y,Z given by extending the trajectory vector by the fly_distance
if direction == "west":
    start_point = spacecraft_point + (perpendicular_nadir * fly_distance)
    end_point = spacecraft_point - (perpendicular_nadir * fly_distance)
elif direction == "east":
    start_point = spacecraft_point - (perpendicular_nadir * fly_distance)
    end_point = spacecraft_point + (perpendicular_nadir * fly_distance)
else:
    print("Direction " + direction + " not defined")

# Generate X, Y, Z some number of points from start and end
num_points = 10000
x_line_space = np.linspace(start_point[0], end_point[0], num_points)
y_line_space = np.linspace(start_point[1], end_point[1], num_points)
z_line_space = np.linspace(start_point[2], end_point[2], num_points)


# Zip up X, Y, Z line spaces and compute velocities
positions = np.array(list(zip(x_line_space, y_line_space, z_line_space)))
velocities = np.array([positions[1] - positions[0]] * num_points)
velocities = np.array([normalize(velocity) * speed for velocity in velocities])
times = np.linspace(-time_offset, time_offset, num_points)
times[int((num_points/2) - 1)], positions[int((num_points/2) - 1)], velocities[int((num_points/2) - 1)]

In [ ]:
# Cell examines computed trajectory and look vector against the radii/wireframe
%matplotlib qt

def WireframeSphere(centre=[0.,0.,0.], radius=[1., 1., 1.],
                    n_meridians=20, n_circles_latitude=None):
    """
    Create the arrays of values to plot the wireframe of a sphere.

    Parameters
    ----------
    centre: array like
        A point, defined as an iterable of three numerical values.
    radius: number
        The radius of the sphere.
    n_meridians: int
        The number of meridians to display (circles that pass on both poles).
    n_circles_latitude: int
        The number of horizontal circles (akin to the Equator) to display.
        Notice this includes one for each pole, and defaults to 4 or half
        of the *n_meridians* if the latter is larger.

    Returns
    -------
    sphere_x, sphere_y, sphere_z: arrays
        The arrays with the coordinates of the points to make the wireframe.
        Their shape is (n_meridians, n_circles_latitude).

    Examples
    --------
    >>> fig = plt.figure()
    >>> ax = fig.gca(projection='3d')
    >>> ax.set_aspect("equal")
    >>> sphere = ax.plot_wireframe(*WireframeSphere(), color="r", alpha=0.5)
    >>> fig.show()

    >>> fig = plt.figure()
    >>> ax = fig.gca(projection='3d')
    >>> ax.set_aspect("equal")
    >>> frame_xs, frame_ys, frame_zs = WireframeSphere()
    >>> sphere = ax.plot_wireframe(frame_xs, frame_ys, frame_zs, color="r", alpha=0.5)
    >>> fig.show()
    """
    if n_circles_latitude is None:
        n_circles_latitude = max(n_meridians/2, 4)
    u, v = np.mgrid[0:2*np.pi:n_meridians*1j, 0:np.pi:n_circles_latitude*1j]
    sphere_x = centre[0] + radius[0] * np.cos(u) * np.sin(v)
    sphere_y = centre[1] + radius[1] * np.sin(u) * np.sin(v)
    sphere_z = centre[2] + radius[2] * np.cos(v)
    return sphere_x, sphere_y, sphere_z

fig = plt.figure()
ax = fig.add_subplot(projection = '3d')
ax.set_aspect("auto")

x_line_space = positions[:, 0]
y_line_space = positions[:, 1]
z_line_space = positions[:, 2]

look_vectors = np.array([normalize(position) * 1000 for position in positions])

# Plot sphere as x, y, z
frame_xs, frame_ys, frame_zs = WireframeSphere(radius=mars_radii)
sphere = ax.plot_wireframe(frame_xs, frame_ys, frame_zs, color="r", alpha=0.5)
ax.plot3D(x_line_space, y_line_space, z_line_space, 'gray', marker='o')
# for i, lv in enumerate(look_vectors):
    # ax.plot3D([lv[0], positions[i][0]], [lv[1], positions[i][1]], [lv[2], positions[i][2]], 'orange', marker='o')

# Plot radii and center point in ECEF
# ax.plot3D(mars_radii[0], 0, 0, 'blue', marker='o')
# ax.plot3D(0, mars_radii[1], 0, 'orange', marker='o')
# ax.plot3D(0, 0, mars_radii[2], 'green', marker='o')
# ax.plot3D(0, 0, 0, 'black', marker='o')

fig.show()

In [ ]:
spk_file = "/Path/to/custom_clipper_path.spk"
if Path(spk_file).is_file():
    os.remove(spk_file)

driver = ClipperEISWACPBIsisLabelNaifSpiceDriver(clipper_cub, props={"nadir": False, "kernels": eis_kernels})
with driver as active_driver:
    ephem_start = active_driver.ephemeris_start_time
ephemeris_time = [time + ephem_start for time in times]

handle = spice.spkopn(spk_file, "SPK", 512)
states = pyspiceql.concatStates(positions, velocities)

spice.spkw13(handle,
            -159,
            499,
            "IAU_MARS",
            ephemeris_time[0],
            ephemeris_time[-1],
            "Custom Trajectory",
            3,
            len(ephemeris_time),
            states,
            ephemeris_time)

spice.spkcls(handle)
eis_kernels.append(spk_file)

In [ ]:
def interp_3d(time, xyz_array, times):
    f1 = np.interp(time, times, xyz_array[:, 0])
    f2 = np.interp(time, times, xyz_array[:, 1])
    f3 = np.interp(time, times, xyz_array[:, 2])

    return np.array([f1, f2, f3])

def compute_segments_with_trajectory(ts,
                                     radii, 
                                     IFOV, 
                                     block_size, 
                                     smear_fraction, 
                                     tset, 
                                     hmax, 
                                     positions, 
                                     velocities, 
                                     times):
    segment_data = []
    
    # Setup initial conditions
    position = interp_3d(ts, positions, times)
    velocity = interp_3d(ts, velocities, times)
    r = np.linalg.norm(position)
    h = r - radii
    
    while (h > hmax):
        ts = ts + 0.5
        position = interp_3d(ts, positions, times)
        velocity = interp_3d(ts, velocities, times)
        r = np.linalg.norm(position)
        h = r - radii
    
    while h < hmax:
        direction = math.copysign(1, ts)
        position = interp_3d(ts, positions, times)
        velocity = interp_3d(ts, velocities, times)
        r = np.linalg.norm(position)
        h = r - radii
        pixwid = IFOV * h
        v = np.linalg.norm(velocity)
        unitR = normalize(position)
        unitC = normalize(np.cross(position, velocity))
        unitI = normalize(np.cross(unitC, unitR))
        vg = np.dot(velocity, unitI) * (radii/r)
        lts = pixwid / vg
        ltm = lts * (1 + direction * smear_fraction)
        Nb = 0
        smear = smear_fraction
    
        while smear <= smear_fraction:
            Nb = Nb + 1
            te = ts + ((Nb * block_size) - 1) * ltm
            position = interp_3d(te, positions, times)
            velocity = interp_3d(te, velocities, times)
            r = np.linalg.norm(position)
            v = np.linalg.norm(velocity)
            h = r - radii
            pixwid = IFOV * h
            unitR = normalize(position)
            unitC = normalize(np.cross(position, velocity))
            unitI = normalize(np.cross(unitC, unitR))
            vg = np.dot(velocity, unitI) * (radii/r)
            
            lte = pixwid / vg
            smear = abs(lte - ltm) / ltm
        
        Nb = max(1, Nb - 1)        
        te = ts + ((Nb * block_size) - 1) * ltm
        data_set = [ts, te, Nb]
        segment_data.append(data_set)
        ts = te + tset

    return segment_data

def compute_segments_const_velocity(ts, radii, speed, IFOV, block_size, smear_fraction, tset, height, hmax):
    segment_data = []
    
    v = speed
    s = v * ts
    r0 = radii + height
    r = math.sqrt(r0**2 + s**2)
    h = r - radii
    
    while (h > hmax):
        ts = ts + 0.5
        s = v * ts
        r = math.sqrt(r0**2 + s**2)
        h = r - radii
    
    while h < hmax:
        direction = math.copysign(1, ts)
        s = v * ts
        r = math.sqrt(r0**2 + s**2)
        h = r - radii
        pixwid = IFOV * h
        vg = v * (radii/r0) / (1 + math.pow((s/r0), 2))
        lts = pixwid / vg
        ltm = lts * (1 + direction * smear_fraction)
        Nb = 0
        smear = smear_fraction
        while smear <= smear_fraction:
            Nb = Nb + 1
            te = ts + ((Nb * block_size) - 1) * ltm
            s = v * te
            r = math.sqrt(r0**2 + s**2)
            h  = r - radii
            pixwid = IFOV * h
            vg = v * (radii/r0) / (1 + math.pow((s/r0), 2))
            lte = pixwid / vg
            smear = abs(lte - ltm) / ltm

        Nb = max(1, Nb - 1)
        te = ts + ((Nb * block_size) - 1) * ltm
        data_set = [ts, te, Nb]
        segment_data.append(data_set)
        ts = te + tset
    return segment_data

R = mars_radii[0]
IFOV = 2.18e-4
block_size = 256
f = 0.05
tset = 0.3
hmax = 1000 * scale

segments_from_trajectory = compute_segments_with_trajectory(-time_offset, R, IFOV, block_size, f, tset, hmax, positions, velocities, times)
segments_from_const_velocity = compute_segments_const_velocity(-time_offset, R, speed, IFOV, block_size, f, tset, height, hmax)

In [ ]:
# Define the fore, nadir, and aft sensor lines
detector_offsets = [0, 1023, 2047]

In [ ]:
eis_kernels

# Steps for creating map2cam data

There are three fill cubes used to minimize on disk data for creating the fill data for the generated cubes. There is a left, center and right hill shade cube used to generate the data. Here were the ISIS commands used to generate each.

### Left

```bash
map2map from=/Volumes/workDrive/isis_data/base/dems/molaMarsPlanetaryRadius0005.cub map=$ISISROOT/appdata/templates/maps/sinusoidal.map to=/Volumes/workDrive/molaLeftCrop.cub matchmap=FALSE pixres=from defaultrange=FROM minlat=-40 maxlat=32 minlon=2 maxlon=37 trim=FALSE interp=CUBICCONVOLUTION
```

```bash
shade from=/Volumes/workDrive/molaLeftCrop.cub to=/Volumes/workDrive/molaLeftShade.cub azimuth=90.0 zenith=45.0
```

```bash
map2map from=/Volumes/workDrive/molaLeftShade.cub map=$ISISROOT/appdata/templates/maps/sinusoidal.map to=/Volumes/workDrive/molaLeftShade.resamp.cub matchmap=FALSE pixres=mpp resolution=60 defaultrange=FROM trim=FALSE interp=CUBICCONVOLUTION
```

### Center

```bash
map2map from=/Volumes/workDrive/isis_data/base/dems/molaMarsPlanetaryRadius0005.cub map=$ISISROOT/appdata/templates/maps/sinusoidal.map to=/Volumes/workDrive/molaCenterCrop.cub matchmap=FALSE pixres=from defaultrange=FROM minlat=-11 maxlat=2.5 minlon=45 maxlon=107 trim=FALSE interp=CUBICCONVOLUTION
```

```bash
shade from=/Volumes/workDrive/molaCenterCrop.cub to=/Volumes/workDrive/molaCenterShade.cub azimuth=90.0 zenith=45.0
```

```bash
map2map from=/Volumes/workDrive/molaCenterShade.cub map=$ISISROOT/appdata/templates/maps/sinusoidal.map to=/Volumes/workDrive/molaCenterShade.resamp.cub matchmap=FALSE pixres=mpp resolution=20 defaultrange=FROM trim=FALSE interp=CUBICCONVOLUTION
```

### Right

```bash
map2map from=/Volumes/workDrive/isis_data/base/dems/molaMarsPlanetaryRadius0005.cub map=$ISISROOT/appdata/templates/maps/sinusoidal.map to=/Volumes/workDrive/molaRightCrop.cub matchmap=FALSE pixres=from defaultrange=FROM minlat=-35 maxlat=29 minlon=100 maxlon=147 trim=FALSE interp=CUBICCONVOLUTION
```

```bash
shade from=/Volumes/workDrive/molaRightCrop.cub to=/Volumes/workDrive/molaRightShade.cub azimuth=90.0 zenith=45.0
```

```bash
map2map from=/Volumes/workDrive/molaRightShade.cub map=$ISISROOT/appdata/templates/maps/sinusoidal.map to=/Volumes/workDrive/molaRightShade.resamp.cub matchmap=FALSE pixres=mpp resolution=60 defaultrange=FROM trim=FALSE interp=CUBICCONVOLUTION
```

## Details for generating the data

The original cube used in the map2map command was created from the MOLA DEM under the ISIS base data area. 

If you plan to change the trajectory of the images captured, mainly the latitude and longitude of closest approach you will need to update the shaded images. You'll need to have an idea of the area you are back projecting into your cameras and adjust the min/max lat/lon values in the map2map calls.

When generating the data, the resolution of the left and right hill shades can be left at a lower resolution as the more distant images will have a lower resolution compared to images taken closer to the closest approach.

In [ ]:
segment_path = "/Path/to/folder/to/write/eis_segments/"
fill_path = "/Path/to/folder/with/fill_cubes/"
spk_file = eis_kernels[-1]
ik_file = eis_kernels[5]
sclk_file = eis_kernels[4]
iak_file = eis_kernels[7]
fk_file = eis_kernels[6]
shape_file = os.path.join(isis_data, "base/dems/molaMarsPlanetaryRadius0005.cub")
for detector_offset in detector_offsets:
    i = 0
    driver = ClipperEISWACPBIsisLabelNaifSpiceDriver(clipper_cub, props={"nadir": True, "kernels": eis_kernels})
    with driver as active_driver:
        ephem_start = active_driver.ephemeris_start_time
    
        for segment_data in segments_from_trajectory:
            image_path = os.path.join(segment_path, f"eis_segment_{detector_offset}_{i:02}.cub")
            isis.makecube(to=image_path, 
                          pixels="VALUE", 
                          value=0.0, 
                          samples=4096, 
                          lines=int(segment_data[-1] * block_size), 
                          bands=1)
            isis.copylabel(from_=image_path, 
                           source=clipper_cub, 
                           instrument=False,
                           bandbin=True,
                           kernels=True,
                           mapping=False,
                           radiometry=False,
                           polygon=False,
                           camstats=False)
            try:
                isis.editlab(from_=image_path, options="DELG", grpname="AlphaCube")
            except:
                pass
            isis.editlab(from_=image_path, options="ADDG", grpname="Instrument")
            
            # Most of the following PVL keyword edits are hard coded and may require changes 
            # to the "value" argument to produce the expected/correct label
            isis.editlab(from_=image_path, 
                         options="SETKEY", 
                         grpname="Instrument", 
                         keyword="SpacecraftName", 
                         value="Clipper")
            isis.editlab(from_=image_path, 
                         options="SETKEY", 
                         grpname="Instrument", 
                         keyword="InstrumentId", 
                         value="WAC-PUSHBROOM")
            isis.editlab(from_=image_path, 
                         options="SETKEY", 
                         grpname="Instrument", 
                         keyword="TargetName", 
                         value="Mars")
            start_time = spice.et2utc(ephem_start + segment_data[0], 'ISOC', 14)
            isis.editlab(from_=image_path, 
                         options="SETKEY", 
                         grpname="Instrument", 
                         keyword="StartTime", 
                         value=start_time)
            exposure_duration = segment_data[1] - segment_data[0]
            isis.editlab(from_=image_path, 
                         options="SETKEY", 
                         grpname="Instrument", 
                         keyword="ExposureDuration", 
                         value=exposure_duration/(segment_data[-1] * block_size), units="s")
            isis.editlab(from_=image_path, 
                         options="SETKEY", 
                         grpname="Instrument", 
                         keyword="DetectorOffset", 
                         value=detector_offset)
            
            # Create CSV for time table and write using csv2table
            csv_file_name = "temp.csv"
            with open(csv_file_name, 'w', newline='\n') as csvfile:
                fieldnames = ["EphemerisTime", "ExposureTime", "LineStart"]
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
                writer.writeheader()
                writer.writerow({"EphemerisTime": segment_data[0] + ephem_start, 
                                 "ExposureTime": exposure_duration/(segment_data[-1] * block_size), 
                                 "LineStart": 1})
            isis.csv2table(csv=csv_file_name, 
                           tableName="LineScanTimes", 
                           to=image_path, 
                           coltypes="(Double, Double, Integer)")
            try:
                isis.spiceinit(from_=image_path,
                               spk=spk_file,
                               CKNADIR=True,
                               CKRECON=False,
                               ik=ik_file,
                               sclk=sclk_file,
                               iak=iak_file,
                               fk=fk_file,
                               shape="USER",
                               model=shape_file)
            except subprocess.CalledProcessError as err:
                print('Had an ISIS error:')
                print(' '.join(err.cmd))
                print(err.stdout)
                print(err.stderr)
                raise err
    
            data_image_path = os.path.join(segment_path, f"eis_segment_{detector_offset}_{i:02}.data.cub")
            pixel_data = os.path.join(fill_path, "molaCenterShade.resamp.cub")
            if i <= 19:
                pixel_data = os.path.join(fill_path, "molaRightShade.resamp.cub")
            elif i >= 60:
                pixel_data = os.path.join(fill_path, "molaLeftShade.resamp.cub")
            isis.map2cam(from_=pixel_data,
                         match=image_path,
                         to=data_image_path)
            i += 1

# What to do after the data is generated

## Use eisstitch to combine images

The ISIS app [eisstitch](https://isis.astrogeology.usgs.gov/dev/Application/presentation/Tabbed/eisstitch/eisstitch.html) allows users to combine multiple EIS Clipper pushbroom exposures into a single exposure. Assumptions are made that the cameras internal orientations are the same between exposures.

The cubes passed into eisstitch need to be ordered by the user cronologically, an update to the app could be made so that appropriate time computations could be made regardless of cube order.